# CoralNet ETL parquet comparison

Compare `coralnet_audit_*`, `coralnet_annotations_*`, and `coralnet_images_*` parquets under `s3://dev-datamermaid-sm-sources/etl-outputs/`.

For each file: row counts, key uniqueness metrics, dataset-specific flags, and column names.

## 1. Setup

In [17]:
import os

if "SM_CURRENT_HOST" not in os.environ:
    os.environ["AWS_PROFILE"] = "mermaid-core"

In [18]:
import ibis
import pandas as pd
from great_tables import GT

ibis.options.interactive = True

In [19]:
con = ibis.duckdb.connect()
con.raw_sql("""
    INSTALL httpfs;
    LOAD httpfs;
""")
con.raw_sql("CREATE OR REPLACE SECRET s3 (TYPE S3, PROVIDER CREDENTIAL_CHAIN)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. ETL runs to compare

In [20]:
BUCKET = "dev-datamermaid-sm-sources"
ETL_PREFIX = f"s3://{BUCKET}/etl-outputs"

RUN_SPECS = [
    {
        "label": "20260521_bd1ad74 (etl-outputs root)",
        "run": "20260521_bd1ad74",
        "run_date": "20260521",
        "prefix": f"{ETL_PREFIX}/20260521_bd1ad74",
        "kinds": ("audit", "annotations", "images"),
    },
    {
        "label": "coralnet-etl-dev / 20260521_bd1ad74",
        "run": "20260521_bd1ad74",
        "run_date": "20260521",
        "prefix": f"{ETL_PREFIX}/coralnet-etl-dev/20260521_bd1ad74",
        "kinds": ("audit", "annotations", "images"),
    },
    {
        "label": "coralnet-full-audit / 20260526_807b611",
        "run": "20260526_807b611",
        "run_date": "20260526",
        "prefix": f"{ETL_PREFIX}/coralnet-full-audit/20260526_807b611",
        "kinds": ("audit",),
    },
    {
        "label": "coralnet-remediation / 20260526_807b611",
        "run": "20260526_807b611",
        "run_date": "20260526",
        "prefix": f"{ETL_PREFIX}/coralnet-remediation/20260526_807b611",
        "kinds": ("audit",),
    },
    {
        "label": "coralnet / 20260526_807b611",
        "run": "20260526_807b611",
        "run_date": "20260526",
        "prefix": f"{ETL_PREFIX}/coralnet/20260526_807b611",
        "kinds": ("audit", "annotations", "images"),
    },
]


def parquet_uri(spec: dict, kind: str) -> str:
    return f"{spec['prefix']}/coralnet_{kind}_{spec['run']}.parquet"


def specs_for_kind(kind: str) -> list[dict]:
    return [spec for spec in RUN_SPECS if kind in spec["kinds"]]

## 3. Audit summary

In [21]:
AUDIT_FLAG_COLUMNS = ("is_complete", "image_count_match", "annotations_empty")


def summarize_audit(con, spec: dict) -> dict:
    table = con.read_parquet(parquet_uri(spec, "audit"))
    columns = list(table.columns)

    if "source_id" not in columns:
        raise KeyError(f"source_id not found in {spec['label']}: {columns}")

    missing_flags = [col for col in AUDIT_FLAG_COLUMNS if col not in columns]
    if missing_flags:
        raise KeyError(f"Missing columns in {spec['label']}: {missing_flags}")

    flag_counts = {
        col: table.filter(getattr(table, col)).count().execute() for col in AUDIT_FLAG_COLUMNS
    }

    return {
        "label": spec["label"],
        "run_date": spec["run_date"],
        "row_count": table.count().execute(),
        "unique_source_id": table.source_id.nunique().execute(),
        **flag_counts,
        "n_columns": len(columns),
        "columns": columns,
    }


audit_summaries = [summarize_audit(con, spec) for spec in specs_for_kind("audit")]

audit_summary_df = pd.DataFrame(
    {
        "label": [s["label"] for s in audit_summaries],
        "run_date": [s["run_date"] for s in audit_summaries],
        "row_count": [s["row_count"] for s in audit_summaries],
        "unique_source_id": [s["unique_source_id"] for s in audit_summaries],
        "is_complete": [s["is_complete"] for s in audit_summaries],
        "image_count_match": [s["image_count_match"] for s in audit_summaries],
        "annotations_empty": [s["annotations_empty"] for s in audit_summaries],
        "n_columns": [s["n_columns"] for s in audit_summaries],
    }
)

audit_columns_df = pd.DataFrame(
    [{"label": s["label"], "column": col} for s in audit_summaries for col in s["columns"]]
)

In [22]:
GT(audit_summary_df).tab_header(
    title="CoralNet audit files — row counts, source_id, and completeness flags"
).fmt_number(
    columns=[
        "row_count",
        "unique_source_id",
        "is_complete",
        "image_count_match",
        "annotations_empty",
        "n_columns",
    ],
    decimals=0,
)

GT(_tbl_data=                                     label  run_date  row_count  \
0      20260521_bd1ad74 (etl-outputs root)  20260521       1509   
1      coralnet-etl-dev / 20260521_bd1ad74  20260521       1509   
2   coralnet-full-audit / 20260526_807b611  20260526       1509   
3  coralnet-remediation / 20260526_807b611  20260526          5   
4              coralnet / 20260526_807b611  20260526       1509   

   unique_source_id  is_complete  image_count_match  annotations_empty  \
0              1509         1319               1468                129   
1              1509         1319               1468                129   
2              1509         1335               1470                128   
3                 5            5                  3                  0   
4              1509         1384               1470                123   

   n_columns  
0         24  
1         24  
2         24  
3         24  
4         24  , _body=<great_tables._gt_data.Body object at 0x1353f5eb0>, _boxhead=Boxhead([ColInfo(var='label', type=<ColInfoTypeEnum.default: 1>, column_label='label', column_align='left', column_width=None), ColInfo(var='run_date', type=<ColInfoTypeEnum.default: 1>, column_label='run_date', column_align='right', column_width=None), ColInfo(var='row_count', type=<ColInfoTypeEnum.default: 1>, column_label='row_count', column_align='right', column_width=None), ColInfo(var='unique_source_id', type=<ColInfoTypeEnum.default: 1>, column_label='unique_source_id', column_align='right', column_width=None), ColInfo(var='is_complete', type=<ColInfoTypeEnum.default: 1>, column_label='is_complete', column_align='right', column_width=None), ColInfo(var='image_count_match', type=<ColInfoTypeEnum.default: 1>, column_label='image_count_match', column_align='right', column_width=None), ColInfo(var='annotations_empty', type=<ColInfoTypeEnum.default: 1>, column_label='annotations_empty', column_align='right', column_width=None), ColInfo(var='n_columns', type=<ColInfoTypeEnum.default: 1>, column_label='n_columns', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1353f6030>, _spanners=Spanners([]), _heading=Heading(title='CoralNet audit files — row counts, source_id, and completeness flags', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1353f7680>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1353f6f60>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1353f7440>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1353f5dc0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=Opt

### Audit column names

In [23]:
GT(audit_columns_df).tab_header(title="Audit parquet column names")

GT(_tbl_data=                                   label                    column
0    20260521_bd1ad74 (etl-outputs root)                 source_id
1    20260521_bd1ad74 (etl-outputs root)           audit_timestamp
2    20260521_bd1ad74 (etl-outputs root)         has_images_folder
3    20260521_bd1ad74 (etl-outputs root)       has_annotations_csv
4    20260521_bd1ad74 (etl-outputs root)        has_image_list_csv
..                                   ...                       ...
115          coralnet / 20260526_807b611  labelset_csv_read_failed
116          coralnet / 20260526_807b611  metadata_csv_read_failed
117          coralnet / 20260526_807b611               is_complete
118          coralnet / 20260526_807b611         image_count_match
119          coralnet / 20260526_807b611                    errors

[120 rows x 2 columns], _body=<great_tables._gt_data.Body object at 0x1353f8dd0>, _boxhead=Boxhead([ColInfo(var='label', type=<ColInfoTypeEnum.default: 1>, column_label='label', column_align='left', column_width=None), ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1353f5be0>, _spanners=Spanners([]), _heading=Heading(title='Audit parquet column names', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x13522e780>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1353f8e90>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1353f8a40>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_b

## 5. Annotations summary

In [24]:
ANNOTATION_STATUS_VALUES = ("Confirmed", "Unconfirmed", "Unclassified")


def summarize_annotations(con, spec: dict) -> dict:
    table = con.read_parquet(parquet_uri(spec, "annotations"))
    columns = list(table.columns)

    if "source_id" not in columns:
        raise KeyError(f"source_id not found in {spec['label']}: {columns}")

    unique_images = table.select("source_id", "image_id").distinct().count().execute()

    status_counts = {}
    if "status" in columns:
        for status in ANNOTATION_STATUS_VALUES:
            status_counts[f"status_{status.lower()}"] = (
                table.filter(table.status == status).count().execute()
            )
        status_counts["status_null"] = table.filter(table.status.isnull()).count().execute()  # noqa: PD003

    return {
        "label": spec["label"],
        "run_date": spec["run_date"],
        "row_count": table.count().execute(),
        "unique_source_id": table.source_id.nunique().execute(),
        "unique_source_image": unique_images,
        **status_counts,
        "n_columns": len(columns),
        "columns": columns,
    }


annotation_summaries = [summarize_annotations(con, spec) for spec in specs_for_kind("annotations")]

annotation_summary_df = pd.DataFrame(annotation_summaries).drop(columns=["columns"])

annotation_columns_df = pd.DataFrame(
    [{"label": s["label"], "column": col} for s in annotation_summaries for col in s["columns"]]
)

In [25]:
GT(annotation_summary_df).tab_header(
    title="CoralNet annotations files — row counts, source_id, and status"
).fmt_number(
    columns=[col for col in annotation_summary_df.columns if col not in {"label", "run_date"}],
    decimals=0,
)

GT(_tbl_data=                                 label  run_date  row_count  unique_source_id  \
0  20260521_bd1ad74 (etl-outputs root)  20260521   21930990              1306   
1  coralnet-etl-dev / 20260521_bd1ad74  20260521   21930990              1306   
2          coralnet / 20260526_807b611  20260526   20385626              1294   

   unique_source_image  status_confirmed  status_unconfirmed  \
0               539174          15893411             6031999   
1               539174          15893411             6031999   
2               528564          20385626                   0   

   status_unclassified  status_null  n_columns  
0                 5580            0          6  
1                 5580            0          6  
2                    0            0          6  , _body=<great_tables._gt_data.Body object at 0x1353f99a0>, _boxhead=Boxhead([ColInfo(var='label', type=<ColInfoTypeEnum.default: 1>, column_label='label', column_align='left', column_width=None), ColInfo(var='run_date', type=<ColInfoTypeEnum.default: 1>, column_label='run_date', column_align='right', column_width=None), ColInfo(var='row_count', type=<ColInfoTypeEnum.default: 1>, column_label='row_count', column_align='right', column_width=None), ColInfo(var='unique_source_id', type=<ColInfoTypeEnum.default: 1>, column_label='unique_source_id', column_align='right', column_width=None), ColInfo(var='unique_source_image', type=<ColInfoTypeEnum.default: 1>, column_label='unique_source_image', column_align='right', column_width=None), ColInfo(var='status_confirmed', type=<ColInfoTypeEnum.default: 1>, column_label='status_confirmed', column_align='right', column_width=None), ColInfo(var='status_unconfirmed', type=<ColInfoTypeEnum.default: 1>, column_label='status_unconfirmed', column_align='right', column_width=None), ColInfo(var='status_unclassified', type=<ColInfoTypeEnum.default: 1>, column_label='status_unclassified', column_align='right', column_width=None), ColInfo(var='status_null', type=<ColInfoTypeEnum.default: 1>, column_label='status_null', column_align='right', column_width=None), ColInfo(var='n_columns', type=<ColInfoTypeEnum.default: 1>, column_label='n_columns', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1353faba0>, _spanners=Spanners([]), _heading=Heading(title='CoralNet annotations files — row counts, source_id, and status', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1353fa6f0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1353fa810>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1353fa840>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1353fa960>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', val

### Annotations column names

In [26]:
GT(annotation_columns_df).tab_header(title="Annotations parquet column names")

GT(_tbl_data=                                  label       column
0   20260521_bd1ad74 (etl-outputs root)    source_id
1   20260521_bd1ad74 (etl-outputs root)     image_id
2   20260521_bd1ad74 (etl-outputs root)          row
3   20260521_bd1ad74 (etl-outputs root)          col
4   20260521_bd1ad74 (etl-outputs root)  coralnet_id
5   20260521_bd1ad74 (etl-outputs root)       status
6   coralnet-etl-dev / 20260521_bd1ad74    source_id
7   coralnet-etl-dev / 20260521_bd1ad74     image_id
8   coralnet-etl-dev / 20260521_bd1ad74          row
9   coralnet-etl-dev / 20260521_bd1ad74          col
10  coralnet-etl-dev / 20260521_bd1ad74  coralnet_id
11  coralnet-etl-dev / 20260521_bd1ad74       status
12          coralnet / 20260526_807b611    source_id
13          coralnet / 20260526_807b611     image_id
14          coralnet / 20260526_807b611          row
15          coralnet / 20260526_807b611          col
16          coralnet / 20260526_807b611  coralnet_id
17          coralnet / 20260526_807b611       status, _body=<great_tables._gt_data.Body object at 0x1353fbbc0>, _boxhead=Boxhead([ColInfo(var='label', type=<ColInfoTypeEnum.default: 1>, column_label='label', column_align='left', column_width=None), ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1353faab0>, _spanners=Spanners([]), _heading=Heading(title='Annotations parquet column names', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1353fb2f0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1353fade0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1353facf0>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_botto

## 6. Images summary

In [27]:
IMAGE_FLAG_COLUMNS = ("needs_resize",)
IMAGE_HEADER_STATUSES = (
    "ok",
    "sof_not_found_extended",
    "pil_fallback",
    "corrupt",
    "not_found",
    "s3_error",
    "worker_exception",
)


def summarize_images(con, spec: dict) -> dict:
    table = con.read_parquet(parquet_uri(spec, "images"))
    columns = list(table.columns)

    if "source_id" not in columns:
        raise KeyError(f"source_id not found in {spec['label']}: {columns}")

    missing_flags = [col for col in IMAGE_FLAG_COLUMNS if col not in columns]
    if missing_flags:
        raise KeyError(f"Missing columns in {spec['label']}: {missing_flags}")

    flag_counts = {
        col: table.filter(getattr(table, col)).count().execute() for col in IMAGE_FLAG_COLUMNS
    }

    null_width = table.filter(table.width.isnull()).count().execute()  # noqa: PD003

    header_counts = {}
    if "header_status" in columns:
        for status in IMAGE_HEADER_STATUSES:
            header_counts[f"header_{status}"] = (
                table.filter(table.header_status == status).count().execute()
            )

    return {
        "label": spec["label"],
        "run_date": spec["run_date"],
        "row_count": table.count().execute(),
        "unique_source_id": table.source_id.nunique().execute(),
        "unique_source_image": table.select("source_id", "image_id").distinct().count().execute(),
        **flag_counts,
        "null_width": null_width,
        **header_counts,
        "n_columns": len(columns),
        "columns": columns,
    }


image_summaries = [summarize_images(con, spec) for spec in specs_for_kind("images")]

image_summary_df = pd.DataFrame(image_summaries).drop(columns=["columns"])

image_columns_df = pd.DataFrame(
    [{"label": s["label"], "column": col} for s in image_summaries for col in s["columns"]]
)

In [28]:
GT(image_summary_df).tab_header(
    title="CoralNet images files — row counts, source_id, and image flags"
).fmt_number(
    columns=[col for col in image_summary_df.columns if col not in {"label", "run_date"}],
    decimals=0,
)

GT(_tbl_data=                                 label  run_date  row_count  unique_source_id  \
0  20260521_bd1ad74 (etl-outputs root)  20260521     539174              1306   
1  coralnet-etl-dev / 20260521_bd1ad74  20260521     539174              1306   
2          coralnet / 20260526_807b611  20260526     528564              1294   

   unique_source_image  needs_resize  null_width  header_ok  \
0               539174        474775         278     505778   
1               539174        474775         278     505778   
2               528564        477297        7549     493199   

   header_sof_not_found_extended  header_pil_fallback  header_corrupt  \
0                          17394                15724               1   
1                          17394                15724               1   
2                          13845                13971               0   

   header_not_found  header_s3_error  header_worker_exception  n_columns  
0               277                0                        0         10  
1               277                0                        0         10  
2              7549                0                        0         10  , _body=<great_tables._gt_data.Body object at 0x1353fd880>, _boxhead=Boxhead([ColInfo(var='label', type=<ColInfoTypeEnum.default: 1>, column_label='label', column_align='left', column_width=None), ColInfo(var='run_date', type=<ColInfoTypeEnum.default: 1>, column_label='run_date', column_align='right', column_width=None), ColInfo(var='row_count', type=<ColInfoTypeEnum.default: 1>, column_label='row_count', column_align='right', column_width=None), ColInfo(var='unique_source_id', type=<ColInfoTypeEnum.default: 1>, column_label='unique_source_id', column_align='right', column_width=None), ColInfo(var='unique_source_image', type=<ColInfoTypeEnum.default: 1>, column_label='unique_source_image', column_align='right', column_width=None), ColInfo(var='needs_resize', type=<ColInfoTypeEnum.default: 1>, column_label='needs_resize', column_align='right', column_width=None), ColInfo(var='null_width', type=<ColInfoTypeEnum.default: 1>, column_label='null_width', column_align='right', column_width=None), ColInfo(var='header_ok', type=<ColInfoTypeEnum.default: 1>, column_label='header_ok', column_align='right', column_width=None), ColInfo(var='header_sof_not_found_extended', type=<ColInfoTypeEnum.default: 1>, column_label='header_sof_not_found_extended', column_align='right', column_width=None), ColInfo(var='header_pil_fallback', type=<ColInfoTypeEnum.default: 1>, column_label='header_pil_fallback', column_align='right', column_width=None), ColInfo(var='header_corrupt', type=<ColInfoTypeEnum.default: 1>, column_label='header_corrupt', column_align='right', column_width=None), ColInfo(var='header_not_found', type=<ColInfoTypeEnum.default: 1>, column_label='header_not_found', column_align='right', column_width=None), ColInfo(var='header_s3_error', type=<ColInfoTypeEnum.default: 1>, column_label='header_s3_error', column_align='right', column_width=None), ColInfo(var='header_worker_exception', type=<ColInfoTypeEnum.default: 1>, column_label='header_worker_exception', column_align='right', column_width=None), ColInfo(var='n_columns', type=<ColInfoTypeEnum.default: 1>, column_label='n_columns', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1353fb3b0>, _spanners=Spanners([]), _heading=Heading(title='CoralNet images files — row counts, source_id, and image flags', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1353fd760>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1353fd790>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1353fd850>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1353fd730>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value',

### Images column names

In [29]:
GT(image_columns_df).tab_header(title="Images parquet column names")

GT(_tbl_data=                                  label         column
0   20260521_bd1ad74 (etl-outputs root)      source_id
1   20260521_bd1ad74 (etl-outputs root)       image_id
2   20260521_bd1ad74 (etl-outputs root)         s3_key
3   20260521_bd1ad74 (etl-outputs root)          width
4   20260521_bd1ad74 (etl-outputs root)         height
5   20260521_bd1ad74 (etl-outputs root)   longest_edge
6   20260521_bd1ad74 (etl-outputs root)      file_size
7   20260521_bd1ad74 (etl-outputs root)   needs_resize
8   20260521_bd1ad74 (etl-outputs root)  header_status
9   20260521_bd1ad74 (etl-outputs root)  error_message
10  coralnet-etl-dev / 20260521_bd1ad74      source_id
11  coralnet-etl-dev / 20260521_bd1ad74       image_id
12  coralnet-etl-dev / 20260521_bd1ad74         s3_key
13  coralnet-etl-dev / 20260521_bd1ad74          width
14  coralnet-etl-dev / 20260521_bd1ad74         height
15  coralnet-etl-dev / 20260521_bd1ad74   longest_edge
16  coralnet-etl-dev / 20260521_bd1ad74      file_size
17  coralnet-etl-dev / 20260521_bd1ad74   needs_resize
18  coralnet-etl-dev / 20260521_bd1ad74  header_status
19  coralnet-etl-dev / 20260521_bd1ad74  error_message
20          coralnet / 20260526_807b611      source_id
21          coralnet / 20260526_807b611       image_id
22          coralnet / 20260526_807b611         s3_key
23          coralnet / 20260526_807b611          width
24          coralnet / 20260526_807b611         height
25          coralnet / 20260526_807b611   longest_edge
26          coralnet / 20260526_807b611      file_size
27          coralnet / 20260526_807b611   needs_resize
28          coralnet / 20260526_807b611  header_status
29          coralnet / 20260526_807b611  error_message, _body=<great_tables._gt_data.Body object at 0x1353fe5a0>, _boxhead=Boxhead([ColInfo(var='label', type=<ColInfoTypeEnum.default: 1>, column_label='label', column_align='left', column_width=None), ColInfo(var='column', type=<ColInfoTypeEnum.default: 1>, column_label='column', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1353fe720>, _spanners=Spanners([]), _heading=Heading(title='Images parquet column names', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1353fdee0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1353fdd60>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1353fda30>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', 